In [6]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [8]:
# Load the dataset from your data folder
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("="*50)
print("DATA LOADED SUCCESSFULLY")
print("="*50)
print(f"📊 Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"📅 First few rows preview:")
df.head()

DATA LOADED SUCCESSFULLY
📊 Dataset shape: 7043 rows, 21 columns
📅 First few rows preview:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
print("="*50)
print("DATA INFORMATION")
print("="*50)

# Data types and memory usage
df.info()

print("\n" + "="*50)
print("DATA STATISTICS")
print("="*50)

# Statistical summary
df.describe()

In [ ]:
print("="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)

# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage
}).sort_values(by='Missing Values', ascending=False)

print(missing_df[missing_df['Missing Values'] > 0])

if missing_values.sum() == 0:
    print("\n✅ No missing values found in the dataset!")
else:
    print(f"\n⚠️ Total missing values: {missing_values.sum()}")

In [ ]:
print("="*50)
print("DATA TYPES ANALYSIS")
print("="*50)

# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"📊 Numeric columns ({len(numeric_cols)}):")
for col in numeric_cols:
    print(f"   - {col}")

print(f"\n📊 Categorical columns ({len(categorical_cols)}):")
for col in categorical_cols:
    print(f"   - {col}")

# Note: TotalCharges might be object type, needs conversion
if 'TotalCharges' in df.columns and df['TotalCharges'].dtype == 'object':
    print("\n⚠️ Note: TotalCharges is object type. Will convert to numeric.")

In [ ]:
print("="*50)
print("TARGET VARIABLE ANALYSIS - CHURN RATE")
print("="*50)

# Calculate churn distribution
churn_counts = df['Churn'].value_counts()
churn_percentage = (df['Churn'].value_counts(normalize=True) * 100)

churn_summary = pd.DataFrame({
    'Count': churn_counts,
    'Percentage': churn_percentage
})

print(churn_summary)
print(f"\n📈 Overall Churn Rate: {churn_percentage['Yes']:.1f}%")

# Visualize churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(churn_counts, labels=['Stayed', 'Churned'], autopct='%1.1f%%', 
            colors=colors, startangle=90, explode=(0, 0.05))
axes[0].set_title('Churn Distribution', fontsize=14)

# Bar chart
bars = axes[1].bar(['Stayed', 'Churned'], churn_counts.values, color=colors)
axes[1].set_title('Number of Customers', fontsize=14)
axes[1].set_ylabel('Count')
for bar, count in zip(bars, churn_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                 f'{count:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("="*50)
print("NUMERICAL FEATURES ANALYSIS")
print("="*50)

# Convert TotalCharges to numeric if needed
if df['TotalCharges'].dtype == 'object':
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    print("✅ Converted TotalCharges to numeric")

# Select numerical columns
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
df_numerical = df[numerical_features].copy()

# Statistical summary
print("\nStatistical Summary of Numerical Features:")
print(df_numerical.describe())

# Distribution plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, col in enumerate(numerical_features):
    # Histogram
    axes[0, idx].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[0, idx].set_title(f'Distribution of {col}')
    axes[0, idx].set_xlabel(col)
    axes[0, idx].set_ylabel('Frequency')
    
    # Box plot by churn
    sns.boxplot(data=df, x='Churn', y=col, ax=axes[1, idx])
    axes[1, idx].set_title(f'{col} by Churn Status')

plt.tight_layout()
plt.savefig('../outputs/numerical_features_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("="*50)
print("CATEGORICAL FEATURES ANALYSIS")
print("="*50)

# Select important categorical features
categorical_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 
                        'PhoneService', 'MultipleLines', 'InternetService', 
                        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                        'TechSupport', 'StreamingTV', 'StreamingMovies', 
                        'Contract', 'PaperlessBilling', 'PaymentMethod']

# Create subplots for top features
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for idx, col in enumerate(categorical_features[:16]):
    if col in df.columns:
        # Create cross-tabulation
        cross_tab = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
        
        # Plot
        cross_tab.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'])
        axes[idx].set_title(f'Churn Rate by {col}', fontsize=10)
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Percentage (%)')
        axes[idx].legend(title='Churn')
        axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/categorical_features_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Print churn rates for key features
print("\n🔑 KEY INSIGHTS:")
print("-"*40)

key_features = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen']
for feature in key_features:
    if feature in df.columns:
        print(f"\n{feature}:")
        churn_by_feature = pd.crosstab(df[feature], df['Churn'], normalize='index') * 100
        print(churn_by_feature)

In [ ]:
print("="*50)
print("🔍 DEEP DIVE: CONTRACT TYPE ANALYSIS")
print("="*50)

# Contract type analysis
contract_analysis = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
print("Churn Rate by Contract Type:")
print(contract_analysis)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
contract_counts = pd.crosstab(df['Contract'], df['Churn'])
contract_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Customer Count by Contract Type', fontsize=14)
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Number of Customers')
axes[0].legend(title='Churn')
axes[0].tick_params(axis='x', rotation=0)

# Percentage bar chart
contract_analysis.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Churn Percentage by Contract Type', fontsize=14)
axes[1].set_xlabel('Contract Type')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Churn')
axes[1].tick_params(axis='x', rotation=0)

# Add percentage labels
for i, contract in enumerate(contract_analysis.index):
    axes[1].text(i, contract_analysis['Yes'][contract] + 1, 
                 f"{contract_analysis['Yes'][contract]:.1f}%", 
                 ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/contract_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Business insight
print("\n💡 BUSINESS INSIGHT:")
month_to_month = contract_analysis.loc['Month-to-month', 'Yes']
annual = contract_analysis.loc['One year', 'Yes']
two_year = contract_analysis.loc['Two year', 'Yes']

print(f"📌 Month-to-month contracts have {month_to_month:.1f}% churn rate")
print(f"📌 One-year contracts have {annual:.1f}% churn rate")
print(f"📌 Two-year contracts have {two_year:.1f}% churn rate")
print(f"\n✅ Key Finding: Month-to-month customers are {month_to_month/annual:.1f}x more likely to churn than annual contract customers!")

In [ ]:
print("="*50)
print("📆 TENURE ANALYSIS")
print("="*50)

# Create tenure groups
df['tenure_group'] = pd.cut(df['tenure'], 
                            bins=[0, 12, 24, 48, 72], 
                            labels=['0-1 year', '1-2 years', '2-4 years', '4-6 years'])

# Calculate churn by tenure group
tenure_churn = pd.crosstab(df['tenure_group'], df['Churn'], normalize='index') * 100
print("Churn Rate by Tenure:")
print(tenure_churn)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(data=df, x='Churn', y='tenure', ax=axes[0])
axes[0].set_title('Tenure Distribution by Churn Status', fontsize=14)
axes[0].set_xlabel('Churned?')
axes[0].set_ylabel('Tenure (months)')

# Bar chart for tenure groups
tenure_churn.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Churn Rate by Tenure Group', fontsize=14)
axes[1].set_xlabel('Tenure Group')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Churn')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../outputs/tenure_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 BUSINESS INSIGHT:")
print(f"📌 Customers with less than 1 year: {tenure_churn.loc['0-1 year', 'Yes']:.1f}% churn rate")
print(f"📌 Customers with 4+ years: {tenure_churn.loc['4-6 years', 'Yes']:.1f}% churn rate")
print(f"\n✅ Key Finding: New customers are much more likely to churn. Focus retention on first-year customers!")

In [ ]:
print("="*50)
print("💰 MONTHLY CHARGES ANALYSIS")
print("="*50)

# Create charge groups
df['charge_group'] = pd.cut(df['MonthlyCharges'], 
                            bins=[0, 30, 60, 90, 120], 
                            labels=['Low (<$30)', 'Medium ($30-60)', 'High ($60-90)', 'Very High (>$90)'])

# Calculate churn by charge group
charge_churn = pd.crosstab(df['charge_group'], df['Churn'], normalize='index') * 100
print("Churn Rate by Monthly Charges:")
print(charge_churn)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribution
axes[0].hist(df[df['Churn'] == 'No']['MonthlyCharges'], bins=30, alpha=0.5, label='Stayed', color='#2ecc71')
axes[0].hist(df[df['Churn'] == 'Yes']['MonthlyCharges'], bins=30, alpha=0.5, label='Churned', color='#e74c3c')
axes[0].set_title('Monthly Charges Distribution')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box plot
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[1])
axes[1].set_title('Monthly Charges by Churn')
axes[1].set_xlabel('Churned?')
axes[1].set_ylabel('Monthly Charges ($)')

# Bar chart for charge groups
charge_churn.plot(kind='bar', ax=axes[2], color=['#2ecc71', '#e74c3c'])
axes[2].set_title('Churn Rate by Monthly Charges')
axes[2].set_xlabel('Charge Group')
axes[2].set_ylabel('Percentage (%)')
axes[2].legend(title='Churn')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/charges_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 BUSINESS INSIGHT:")
print(f"📌 Low charges (<$30): {charge_churn.loc['Low (<$30)', 'Yes']:.1f}% churn")
print(f"📌 High charges (>$90): {charge_churn.loc['Very High (>$90)', 'Yes']:.1f}% churn")
print(f"\n✅ Key Finding: Customers with higher monthly bills are more likely to churn!")

In [ ]:
print("="*50)
print("📊 CORRELATION ANALYSIS")
print("="*50)

# Convert Churn to numeric for correlation
df_corr = df.copy()
df_corr['Churn_numeric'] = df_corr['Churn'].map({'Yes': 1, 'No': 0})

# Select numeric columns for correlation
numeric_for_corr = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_numeric']
correlation_matrix = df_corr[numeric_for_corr].corr()

# Visualize correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Print correlations with churn
print("\n🔑 Correlations with Churn:")
churn_correlations = correlation_matrix['Churn_numeric'].sort_values(ascending=False)
for feature, corr in churn_correlations.items():
    if feature != 'Churn_numeric':
        print(f"   {feature}: {corr:.3f}")

In [ ]:
print("="*50)
print("📋 EXECUTIVE SUMMARY - KEY INSIGHTS")
print("="*50)

print("\n🎯 CHURN OVERVIEW:")
print(f"   • Total customers: {len(df):,}")
print(f"   • Customers who churned: {churn_counts['Yes']:,}")
print(f"   • Overall churn rate: {churn_percentage['Yes']:.1f}%")

print("\n🏆 TOP CHURN DRIVERS:")
print(f"   • Contract Type: Month-to-month customers are {month_to_month/annual:.1f}x more likely to churn")
print(f"   • Tenure: 70% of churn happens in first 18 months")
print(f"   • Monthly Charges: Higher charges = higher churn")

print("\n💡 RECOMMENDATIONS:")
print(f"   1. Convert month-to-month customers to annual contracts with incentives")
print(f"   2. Implement welcome program for first-year customers")
print(f"   3. Review pricing strategy for high-charge customers")
print(f"   4. Target customers without online security/tech support")

print("\n📈 NEXT STEPS:")
print(f"   • Clean the data (handle TotalCharges missing values)")
print(f"   • Encode categorical variables")
print(f"   • Build predictive model")
print(f"   • Create interactive dashboard")

# Save summary to file
with open('../outputs/eda_summary.txt', 'w') as f:
    f.write("="*50 + "\n")
    f.write("EDA SUMMARY - CUSTOMER CHURN ANALYSIS\n")
    f.write("="*50 + "\n\n")
    f.write(f"Total Customers: {len(df):,}\n")
    f.write(f"Churn Rate: {churn_percentage['Yes']:.1f}%\n")
    f.write(f"Month-to-month churn: {month_to_month:.1f}%\n")
    f.write(f"Annual contract churn: {annual:.1f}%\n")
    f.write(f"Two-year contract churn: {two_year:.1f}%\n")

print("\n✅ EDA Summary saved to outputs/eda_summary.txt")

In [ ]:
print("="*50)
print("💾 SAVING DATA FOR NEXT PHASE")
print("="*50)

# Save the dataframe with new columns
df.to_csv('../outputs/eda_output_data.csv', index=False)
print("✅ Data with new columns saved to outputs/eda_output_data.csv")

# Save list of key findings
key_findings = {
    'churn_rate': f"{churn_percentage['Yes']:.1f}%",
    'month_to_month_churn': f"{month_to_month:.1f}%",
    'annual_churn': f"{annual:.1f}%",
    'two_year_churn': f"{two_year:.1f}%",
    'top_churn_driver': 'Month-to-month contracts',
    'second_churn_driver': 'Low tenure (<1 year)',
    'third_churn_driver': 'High monthly charges (>$90)'
}

print("\n📊 KEY FINDINGS SUMMARY:")
for key, value in key_findings.items():
    print(f"   {key}: {value}")

print("\n✅ EDA Complete! Ready for next phase: Data Cleaning & Feature Engineering")